# 🏥 Apollo Voice Engine - AudioLLM Training

**Train the Unified Speech-to-Speech Model on IndicVoices Dataset**

This notebook trains the AudioLLM (Extended Sarvam-1) on speech-to-speech tasks using:
- **SNAC** for audio tokenization (encode/decode)
- **Sarvam-1 2B** as the base LLM with extended audio vocabulary
- **IndicVoices** dataset for Hindi, Tamil, Telugu, Kannada

## Training Phases:
1. **Understanding**: Audio → Text (ASR capability)
2. **Generation**: Text → Audio (TTS capability)
3. **End-to-End**: Audio → Audio (Full speech-to-speech)

---

## 1️⃣ Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate bitsandbytes
!pip install -q snac librosa soundfile
!pip install -q tqdm

print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.4 MB/s eta 0:00:00:00:0100:01
✅ Dependencies installed!


In [3]:
import os
import json
import random
from pathlib import Path
from typing import List, Optional, Tuple
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
from tqdm.notebook import tqdm
import librosa

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from snac import SNAC

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cpu
CUDA available: False


## 2️⃣ Upload Dataset

Upload your `indic_voices_dataset` folder containing:
- `metadata.json`
- `dataset_audio/` folder with audio files

In [4]:
# Option 1: Upload from local machine
from google.colab import files

# Upload the dataset zip file
print("Please upload your indic_voices_dataset.zip file:")
uploaded = files.upload()

Please upload your indic_voices_dataset.zip file:


KeyboardInterrupt: 

In [ ]:
# Unzip the dataset
!unzip -q indic_voices_dataset.zip -d .
!ls indic_voices_dataset/

In [ ]:
# Option 2: Mount Google Drive (if dataset is there)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/indic_voices_dataset' .

## 3️⃣ Configuration

In [ ]:
@dataclass
class TrainingConfig:
    """Training configuration."""
    # Dataset
    dataset_path: str = "indic_voices_dataset"
    metadata_file: str = "metadata.json"
    
    # Model
    model_name: str = "sarvamai/sarvam-1"
    snac_model: str = "hubertsiuzdak/snac_24khz"
    
    # Training
    batch_size: int = 2
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    max_epochs: int = 5
    max_audio_length: int = 10  # seconds
    
    # Hardware
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    mixed_precision: bool = True
    
    # Checkpointing
    checkpoint_dir: str = "checkpoints"
    save_every_n_steps: int = 100
    
    # Languages
    languages: List[str] = None
    
    # Audio vocabulary extension
    AUDIO_TOKEN_OFFSET: int = 50000
    AUDIO_VOCAB_SIZE: int = 4096
    AUDIO_START_TOKEN: str = "<|audio_start|>"
    AUDIO_END_TOKEN: str = "<|audio_end|>"
    
    def __post_init__(self):
        if self.languages is None:
            self.languages = ["hi", "ta", "te", "kn"]

config = TrainingConfig()
print(f"Config: {config}")

## 4️⃣ Load Models

In [ ]:
# Load SNAC audio codec
print("Loading SNAC audio codec...")
snac = SNAC.from_pretrained(config.snac_model)
snac = snac.to(config.device)
snac.eval()
print("✅ SNAC loaded!")

In [ ]:
# Load Sarvam-1 LLM with extended vocabulary
print("Loading Sarvam-1 LLM...")

tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)

# Add special audio tokens
special_tokens = {
    "additional_special_tokens": [
        config.AUDIO_START_TOKEN,
        config.AUDIO_END_TOKEN,
    ]
}
tokenizer.add_special_tokens(special_tokens)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Store original vocab size
original_vocab_size = model.config.vocab_size

# Extend vocabulary for audio tokens
extended_vocab_size = original_vocab_size + config.AUDIO_VOCAB_SIZE + 2  # +2 for start/end
model.resize_token_embeddings(extended_vocab_size)

print(f"✅ Model loaded! Vocab extended: {original_vocab_size} → {extended_vocab_size}")

## 5️⃣ Dataset

In [ ]:
class IndicVoicesDataset(Dataset):
    """Dataset for IndicVoices data."""
    
    def __init__(
        self,
        config: TrainingConfig,
        snac_model,
        phase: str = "understanding",
        split: str = "train",
        split_ratio: float = 0.9
    ):
        self.config = config
        self.snac = snac_model
        self.phase = phase
        self.split = split
        
        # Load metadata
        metadata_path = Path(config.dataset_path) / config.metadata_file
        with open(metadata_path, 'r', encoding='utf-8') as f:
            self.all_samples = json.load(f)
        
        # Filter by languages
        self.samples = [
            s for s in self.all_samples 
            if s['language'] in config.languages
        ]
        
        # Split train/val
        random.seed(42)
        random.shuffle(self.samples)
        split_idx = int(len(self.samples) * split_ratio)
        
        if split == "train":
            self.samples = self.samples[:split_idx]
        else:
            self.samples = self.samples[split_idx:]
        
        # Filter out very short/long samples
        self.samples = [
            s for s in self.samples 
            if 0.5 <= s['duration'] <= config.max_audio_length
        ]
        
        print(f"Loaded {len(self.samples)} samples for {split} split")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Load audio
        audio_path = Path(self.config.dataset_path) / sample['audio_path']
        audio = self._load_audio(audio_path, sample['sampling_rate'])
        
        # Encode audio with SNAC
        with torch.no_grad():
            audio_tokens = self._encode_audio(audio)
        
        return {
            'audio_tokens': audio_tokens,
            'text': sample['text'],
            'language': sample['language']
        }
    
    def _load_audio(self, path: Path, original_sr: int) -> torch.Tensor:
        """Load and resample audio to 24kHz for SNAC."""
        try:
            audio, sr = librosa.load(path, sr=original_sr)
            if sr != 24000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=24000)
            audio_tensor = torch.from_numpy(audio).float().unsqueeze(0)
            return audio_tensor.to(self.config.device)
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return torch.zeros(1, 24000, device=self.config.device)
    
    def _encode_audio(self, audio: torch.Tensor) -> torch.Tensor:
        """Encode audio with SNAC and flatten."""
        codes = self.snac.encode(audio)
        # Flatten multi-scale codes
        flattened = self._flatten_codes(codes)
        # Apply offset
        return flattened + self.config.AUDIO_TOKEN_OFFSET
    
    def _flatten_codes(self, codes):
        """Flatten multi-scale SNAC codes."""
        coarse, medium, fine = codes
        batch_size = coarse.shape[0]
        num_frames = coarse.shape[1]
        
        medium_per_coarse = medium.shape[1] // num_frames if num_frames > 0 else 1
        fine_per_coarse = fine.shape[1] // num_frames if num_frames > 0 else 1
        
        flattened = []
        for i in range(num_frames):
            flattened.append(coarse[:, i:i+1])
            m_start, m_end = i * medium_per_coarse, (i + 1) * medium_per_coarse
            flattened.append(medium[:, m_start:m_end])
            f_start, f_end = i * fine_per_coarse, (i + 1) * fine_per_coarse
            flattened.append(fine[:, f_start:f_end])
        
        if flattened:
            return torch.cat(flattened, dim=1).squeeze(0)
        return torch.tensor([], device=coarse.device)

# Test dataset
train_dataset = IndicVoicesDataset(config, snac, phase="understanding", split="train")
val_dataset = IndicVoicesDataset(config, snac, phase="understanding", split="val")

print(f"\nTrain: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")

In [ ]:
# Test loading a sample
sample = train_dataset[0]
print(f"Audio tokens shape: {sample['audio_tokens'].shape}")
print(f"Text: {sample['text'][:100]}...")
print(f"Language: {sample['language']}")

## 6️⃣ Collate Function

In [ ]:
def collate_fn(batch, tokenizer, config, phase="understanding"):
    """Collate function for DataLoader."""
    
    # Get max audio token length
    max_audio_len = max(item['audio_tokens'].shape[0] for item in batch)
    
    input_ids_list = []
    labels_list = []
    
    audio_start_id = tokenizer.convert_tokens_to_ids(config.AUDIO_START_TOKEN)
    audio_end_id = tokenizer.convert_tokens_to_ids(config.AUDIO_END_TOKEN)
    
    for item in batch:
        audio_tokens = item['audio_tokens']
        text = item['text']
        language = item['language']
        
        # Pad audio tokens
        pad_len = max_audio_len - audio_tokens.shape[0]
        if pad_len > 0:
            audio_tokens = torch.cat([
                audio_tokens,
                torch.zeros(pad_len, dtype=audio_tokens.dtype, device=audio_tokens.device)
            ])
        
        if phase == "understanding":
            # Audio → Text (ASR)
            prompt = f"Transcribe the following {language} audio: "
            prompt_tokens = tokenizer(prompt, return_tensors='pt', padding=False)['input_ids'].squeeze(0)
            
            target_tokens = tokenizer(
                text,
                return_tensors='pt',
                padding=False,
                truncation=True,
                max_length=256
            )['input_ids'].squeeze(0)
            
            # Sequence: prompt + <audio_start> + audio + <audio_end> + target
            full_input = torch.cat([
                prompt_tokens.to(audio_tokens.device),
                torch.tensor([audio_start_id], device=audio_tokens.device),
                audio_tokens.long(),
                torch.tensor([audio_end_id], device=audio_tokens.device),
                target_tokens.to(audio_tokens.device)
            ])
            
            # Labels: -100 for non-target, actual tokens for target
            input_len = len(prompt_tokens) + 1 + len(audio_tokens) + 1
            labels = torch.cat([
                torch.full((input_len,), -100, device=audio_tokens.device),
                target_tokens.to(audio_tokens.device)
            ])
            
        else:  # generation
            # Text → Audio (TTS)
            prompt = f"Synthesize in {language}: {text} "
            prompt_tokens = tokenizer(prompt, return_tensors='pt', padding=False)['input_ids'].squeeze(0)
            
            # Sequence: prompt + <audio_start> + audio + <audio_end>
            full_input = torch.cat([
                prompt_tokens.to(audio_tokens.device),
                torch.tensor([audio_start_id], device=audio_tokens.device),
                audio_tokens.long(),
                torch.tensor([audio_end_id], device=audio_tokens.device)
            ])
            
            # Labels: -100 for prompt, actual tokens for audio
            labels = torch.cat([
                torch.full((len(prompt_tokens),), -100, device=audio_tokens.device),
                torch.tensor([audio_start_id], device=audio_tokens.device),
                audio_tokens.long(),
                torch.tensor([audio_end_id], device=audio_tokens.device)
            ])
        
        input_ids_list.append(full_input)
        labels_list.append(labels)
    
    # Pad to same length
    max_len = max(ids.shape[0] for ids in input_ids_list)
    
    padded_input_ids = []
    padded_labels = []
    padded_attention_mask = []
    
    for ids, labs in zip(input_ids_list, labels_list):
        pad_len = max_len - ids.shape[0]
        
        padded_input_ids.append(
            torch.cat([ids, torch.zeros(pad_len, dtype=ids.dtype, device=ids.device)])
        )
        padded_labels.append(
            torch.cat([labs, torch.full((pad_len,), -100, device=labs.device)])
        )
        padded_attention_mask.append(
            torch.cat([torch.ones(ids.shape[0], device=ids.device), torch.zeros(pad_len, device=ids.device)])
        )
    
    return {
        'input_ids': torch.stack(padded_input_ids).cpu(),
        'labels': torch.stack(padded_labels).cpu(),
        'attention_mask': torch.stack(padded_attention_mask).cpu()
    }

## 7️⃣ Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, config, phase):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(loader, desc=f"Training {phase}")
    optimizer.zero_grad()
    
    for batch_idx, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(config.device).long()
        labels = batch['labels'].to(config.device).long()
        attention_mask = batch['attention_mask'].to(config.device)
        
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs.loss / config.gradient_accumulation_steps
            
            scaler.scale(loss).backward()
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss / config.gradient_accumulation_steps
            loss.backward()
        
        if (batch_idx + 1) % config.gradient_accumulation_steps == 0:
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()
        
        total_loss += loss.item() * config.gradient_accumulation_steps
        num_batches += 1
        
        pbar.set_postfix({'loss': total_loss / num_batches})
    
    return total_loss / num_batches

@torch.no_grad()
def validate(model, loader, config):
    """Validate the model."""
    model.eval()
    total_loss = 0.0
    num_batches = 0
    
    for batch in tqdm(loader, desc="Validating"):
        input_ids = batch['input_ids'].to(config.device).long()
        labels = batch['labels'].to(config.device).long()
        attention_mask = batch['attention_mask'].to(config.device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        total_loss += outputs.loss.item()
        num_batches += 1
    
    return total_loss / num_batches

## 8️⃣ Train!

In [ ]:
# Training settings
PHASE = "understanding"  # or "generation"
NUM_EPOCHS = 3

# Create checkpoint directory
os.makedirs(config.checkpoint_dir, exist_ok=True)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda batch: collate_fn(batch, tokenizer, config, PHASE)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: collate_fn(batch, tokenizer, config, PHASE)
)

# Optimizer and scheduler
optimizer = AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

total_steps = len(train_loader) * NUM_EPOCHS // config.gradient_accumulation_steps
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, total_steps))

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler() if config.mixed_precision else None

print(f"\n{'='*60}")
print(f"Starting Training: {PHASE.upper()}")
print(f"{'='*60}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {config.batch_size}")
print(f"Gradient accumulation: {config.gradient_accumulation_steps}")
print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"Total steps: {total_steps}")
print(f"{'='*60}\n")

In [ ]:
# Training loop
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\n📍 Epoch {epoch + 1}/{NUM_EPOCHS}")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, config, PHASE)
    
    # Validate
    val_loss = validate(model, val_loader, config)
    
    print(f"   Train Loss: {train_loss:.4f}")
    print(f"   Val Loss: {val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = os.path.join(config.checkpoint_dir, f"best_{PHASE}.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, checkpoint_path)
        print(f"   ✅ Saved best model: {checkpoint_path}")
    
    # Save epoch checkpoint
    checkpoint_path = os.path.join(config.checkpoint_dir, f"epoch_{epoch+1}_{PHASE}.pt")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss,
    }, checkpoint_path)

print("\n✅ Training complete!")

## 9️⃣ Save & Download Model

In [ ]:
# Save the final model
final_save_path = "apollo_audio_llm_trained"
os.makedirs(final_save_path, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"✅ Model saved to: {final_save_path}")

In [ ]:
# Zip and download
!zip -r apollo_audio_llm_trained.zip apollo_audio_llm_trained/

from google.colab import files
files.download('apollo_audio_llm_trained.zip')

## 🔟 Test the Model

In [ ]:
# Test inference
model.eval()

# Get a sample
sample = val_dataset[0]
audio_tokens = sample['audio_tokens']
true_text = sample['text']

# Prepare input
audio_start_id = tokenizer.convert_tokens_to_ids(config.AUDIO_START_TOKEN)
audio_end_id = tokenizer.convert_tokens_to_ids(config.AUDIO_END_TOKEN)

prompt = f"Transcribe the following {sample['language']} audio: "
prompt_tokens = tokenizer(prompt, return_tensors='pt')['input_ids'].to(config.device)

input_ids = torch.cat([
    prompt_tokens,
    torch.tensor([[audio_start_id]], device=config.device),
    audio_tokens.unsqueeze(0).long(),
    torch.tensor([[audio_end_id]], device=config.device)
], dim=1)

# Generate
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode
generated = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

print(f"True text: {true_text}")
print(f"Generated: {generated}")

---

## 📝 Notes

- **Phase 1 (Understanding)**: Train the model to transcribe audio to text
- **Phase 2 (Generation)**: Train the model to synthesize text to audio
- **For production**: Train for more epochs (10-20) with larger batch sizes
- **Memory issues**: Reduce batch_size or max_audio_length

---

🎉 **Congratulations!** You've trained the Apollo AudioLLM!